# 11_incremental_payment_lifecycle

- Day-7 Incremental Processing. Reads the full event history already sitting in silver.external_payment -(built by 10_silver_external_payment) and produces a SECOND table,
- silver.external_payment_current, holding one row per payment "thread" -
- its latest known state - built with a Delta MERGE so re-running this notebook never duplicates or double-counts anything.

In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import re

### 1. Read the full Silver event history

In [0]:
events_df = spark.table(silver_table("external_payment"))
print(f"Total events: {events_df.count()}")

Total events: 14


### 2. Extract a payment_thread_id
Regex: `PMT` + 3-digit sequence, ignoring the date and any suffix.
Rows that DON'T match this pattern (e.g. a business-key-duplicate
survivor with an odd id format) fall back to using their own
payment_id as a single-event thread - they still get a current-state
row, just with no history to link.

In [0]:
THREAD_ID_PATTERN = r'^(PMT\d{3})\d{8}(\d{3})?$'

threaded_df = events_df.withColumn(
    "payment_thread_id",
    F.when(
        F.col("payment_id").rlike(THREAD_ID_PATTERN),
        F.regexp_extract(F.col("payment_id"), THREAD_ID_PATTERN, 1)
    ).otherwise(F.col("payment_id"))  # fallback: no pattern match -> its own thread
)

threaded_df.select("payment_id", "payment_thread_id", "status", "event_timestamp", "settlement_timestamp").orderBy("payment_thread_id", "event_timestamp").show(30, truncate=False)

+-----------------+-----------------+---------+-------------------+--------------------+
|payment_id       |payment_thread_id|status   |event_timestamp    |settlement_timestamp|
+-----------------+-----------------+---------+-------------------+--------------------+
|PMT00120260915   |PMT001           |INITIATED|2026-09-15 09:15:00|2026-09-15 09:15:00 |
|PMT00120260916   |PMT001           |INITIATED|2026-09-16 09:15:00|2026-09-16 09:15:00 |
|PMT00120260916001|PMT001           |SETTLED  |2026-09-16 13:30:00|2026-09-16 14:00:00 |
|PMT00120260917   |PMT001           |INITIATED|2026-09-17 09:15:00|2026-09-17 10:00:00 |
|PMT00220260915   |PMT002           |SETTLED  |2026-09-15 11:00:00|2026-09-15 11:00:00 |
|PMT00220260916   |PMT002           |SETTLED  |2026-09-16 11:00:00|2026-09-16 11:00:00 |
|PMT00220260917   |PMT002           |SETTLED  |2026-09-17 11:00:00|2026-09-17 11:00:00 |
|PMT00320260915   |PMT003           |SETTLED  |2026-09-15 13:30:00|2026-09-15 13:30:00 |
|PMT00320260916   |PM

### 3. Determine current state per thread
Latest COALESCE(settlement_timestamp, event_timestamp) wins.

In [0]:
w = Window.partitionBy("payment_thread_id").orderBy(
    F.coalesce(F.col("settlement_timestamp"), F.col("event_timestamp")).desc()
)

current_state_df = (
    threaded_df
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .withColumn("thread_event_count", F.lit(None).cast("int"))  # filled in next cell
)

event_counts = threaded_df.groupBy("payment_thread_id").agg(F.count("*").alias("thread_event_count_calc"))
current_state_df = (
    current_state_df.drop("thread_event_count")
    .join(event_counts, on="payment_thread_id", how="left")
    .withColumnRenamed("thread_event_count_calc", "thread_event_count")
)

print(f"Distinct payment threads: {current_state_df.count()}")
current_state_df.select("payment_thread_id", "payment_id", "status", "thread_event_count").orderBy("payment_thread_id").show(truncate=False)

Distinct payment threads: 5
+-----------------+-----------------+---------+------------------+
|payment_thread_id|payment_id       |status   |thread_event_count|
+-----------------+-----------------+---------+------------------+
|PMT001           |PMT00120260917   |INITIATED|4                 |
|PMT002           |PMT00220260917   |SETTLED  |3                 |
|PMT003           |PMT00320260917   |SETTLED  |3                 |
|PMT004           |PMT00420260917   |FAILED   |3                 |
|PMT005           |PMT00520260917001|REVERSED |1                 |
+-----------------+-----------------+---------+------------------+



### 4. MERGE into silver.external_payment_current
Idempotent by design: re-running this notebook against the same
events re-computes the same current-state rows and MERGE updates
them in place - it does not append duplicates.

In [0]:
merge_into_silver(current_state_df, "external_payment_current", merge_key_cols=["payment_thread_id"])
print("Merged into silver.external_payment_current")

Merged into silver.external_payment_current


### 5. Idempotency check
- Run this notebook's  second time and confirm the row count in external_payment_current does NOT
change 

In [0]:
result_df = spark.table(silver_table("external_payment_current"))
print(f"silver.external_payment_current row count: {result_df.count()}")
print("(expect this number to stay identical if you re-run Section 4 again without new source data)")
result_df.select("payment_thread_id", "payment_id", "fund_id", "payment_type", "amount", "status", "thread_event_count").orderBy("payment_thread_id").show(truncate=False)

silver.external_payment_current row count: 5
(expect this number to stay identical if you re-run Section 4 again without new source data)
+-----------------+-----------------+-------+------------+--------+---------+------------------+
|payment_thread_id|payment_id       |fund_id|payment_type|amount  |status   |thread_event_count|
+-----------------+-----------------+-------+------------+--------+---------+------------------+
|PMT001           |PMT00120260917   |FND001 |CAPITAL_CALL|750000.0|INITIATED|4                 |
|PMT002           |PMT00220260917   |FND002 |INVESTMENT  |500000.0|SETTLED  |3                 |
|PMT003           |PMT00320260917   |FND003 |EXPENSE     |42000.0 |SETTLED  |3                 |
|PMT004           |PMT00420260917   |FND001 |DISTRIBUTION|150000.0|FAILED   |3                 |
|PMT005           |PMT00520260917001|FND002 |INVESTMENT  |225000.0|REVERSED |1                 |
+-----------------+-----------------+-------+------------+--------+---------+---------